<a href="https://www.kaggle.com/code/alexvmt/train-and-evaluate-terainet?scriptVersionId=321189515" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Train and evaluate TeraiNet

1. Set variables and parameters
2. Prepare train, val and test datasets
3. Prepare model
4. Train model
5. Evaluate model
6. Log run to W&B

## Setup

### Change working directory

In [ ]:
%cd ../../

### Clone TeraiNet repo

In [ ]:
!git clone -b chore/refactor_training https://github.com/alexvmt/terainet.git

### Imports

Follow [mewc-flow](https://github.com/zaandahl/mewc-flow/blob/main/requirements.txt) for the key package versions

In [ ]:
%pip install keras==3.3.3 kimm==0.2.5 tensorflow==2.16.1

In [ ]:
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]

project_dir = Path("terainet")
sys.path.insert(0, str(project_dir / "src"))

from terainet import (
    build_model,
    create_datasets,
    evaluate_dataset,
    load_training_config,
    log_run_to_wandb,
    prepare_training_data,
    save_evaluation_artifacts,
    set_random_seed,
    train_and_save,
)

### Set variables and parameters

In [ ]:
training_config_path = project_dir / "training.yaml"
config = load_training_config(training_config_path)

# Kaggle-specific authentication remains in the notebook. Source code only logs
# through an already authenticated W&B session.
use_wandb_logging = config.wandb.get("enabled", False)

In [ ]:
# log in to w&b using api key
if use_wandb_logging:
    user_secrets = UserSecretsClient()
    key = user_secrets.get_secret("wandb")
    !wandb login $key

## Prepare train, val and test datasets

### Filter to keep only single-snippet images

In [ ]:
set_random_seed(config.seed)
preparation = prepare_training_data(config)

if preparation.filter_stats:
    for subset, stats in preparation.filter_stats.items():
        print(
            f"{subset}: {stats['original_total']} → {stats['filtered_total']} "
            f"({stats['removed_total']} removed)"
        )

### Sample images

### Create datasets

In [ ]:
datasets = create_datasets(config, preparation)
print(f"Train element spec: {datasets.train.element_spec}")
print(f"Validation element spec: {datasets.validation.element_spec}")
print(f"Test element spec: {datasets.test.element_spec}")

### Resize (and augment) images

## Prepare model

In [ ]:
model_build = build_model(config)
model_build.model.summary(show_trainable=True)

## Train model

Follow [mewc-train](https://github.com/zaandahl/mewc-train) for the training parameters

In [ ]:
run_result = train_and_save(model_build, datasets, config, preparation)
print(f"Training time (minutes): {run_result.training_time_minutes}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(run_result.history.history["loss"], label="Training loss")
plt.plot(run_result.history.history["val_loss"], label="Validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid()
plt.show()

## Evaluate model

### Get predictions on test dataset

In [ ]:
test_result = evaluate_dataset(
    run_result.model,
    datasets.test,
    config.class_names,
    name="test",
)
save_evaluation_artifacts(test_result, config)
print(test_result.metrics)

In [ ]:
ood_result = None
if datasets.ood is not None:
    ood_result = evaluate_dataset(
        run_result.model,
        datasets.ood,
        config.class_names,
        name="ood",
    )
    print(ood_result.metrics)

### Get performance metrics, classification report and confusion matrix

In [ ]:
print(test_result.report)

In [ ]:
print(test_result.confusion_matrix)

In [ ]:
from IPython.display import Image, display

display(Image(filename=config.artifact_paths["confusion_matrix"]))

## Log run to W&B

In [ ]:
if use_wandb_logging:
    log_run_to_wandb(config, run_result, test_result, ood_result)